# Mission 1: 119 신고자 성별 분류, ECAPA-TDNN 임베딩 기반 접근

이 노트북은 baseline부터 최종 채택 모델까지의 전체 실험 과정과, 시도했지만 채택하지 않은
방법들(무엇을 왜 버렸는지 포함)을 순서대로 정리한 기록입니다.
모든 셀은 `Work/Common`, `Work/ProvidedValidation`에 이미 만들어둔 임베딩(shard)을
그대로 불러와서 사용하며, 원본 오디오에서 임베딩을 다시 추출하지는 않습니다
(그 과정은 `Code/03_extract_ecapa_embeddings.py`에 따로 있습니다).


> **참고**: 이 노트북은 `01_extract_training.py` -> `02_common_preprocess.py` ->
> `03_extract_ecapa_embeddings.py` -> `05a~05c`(Data/Validation)를 이미 실행해서
> `Work/Common`, `Work/ProvidedValidation` 아래에 임베딩(shard)이 만들어져 있다는
> 전제 하에 작성되었습니다. 아래 셀들의 출력은 **실제로 로컬에서 한 번 실행한
> 결과가 그대로 저장된 것**이며, 이 상태로 열어서 읽는 것만으로 전체 과정과
> 결과를 확인할 수 있습니다.
>
> 처음부터 다시 실행하려면 위 파이프라인을
> 먼저 실행해야 합니다. 전체 파이프라인 없이 이 노트북만 실행하면 아래 데이터
> 로드 셀에서 안내 메시지와 함께 중단됩니다.


## 1. 문제 정의와 접근 방법

119 신고 통화에서 신고자(speaker=1) 구간만 잘라내어, 그 목소리만으로 성별(M/F)을 분류합니다.
VoxCeleb으로 사전학습된 화자 임베딩 모델 ECAPA-TDNN(frozen)으로 192차원 벡터를 뽑고,
그 위에 가벼운 분류기를 학습시키는 방식을 택했습니다. 성별은 화자 특성에 크게 좌우되는
문제라, 화자 구분용으로 학습된 임베딩 안에 성별 정보도 강하게 녹아있을 것이라는
가설에서 출발했습니다.


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd().parent  # 이 노트북이 Mission1/ 바로 아래에 있다고 가정
TRAIN_SHARD_DIR = BASE_DIR / "Work" / "Common" / "embeddings" / "shards"
PV_SHARD_DIR = BASE_DIR / "Work" / "ProvidedValidation" / "Common" / "embeddings" / "shards"

def load_shards(shard_dir):
    npy_files = sorted(shard_dir.glob("shard_*.npy"))
    csv_files = sorted(shard_dir.glob("shard_*.csv"))
    X_list, meta_list = [], []
    for npy_path, csv_path in zip(npy_files, csv_files):
        x = np.load(npy_path)
        if x.shape[0] == 0:
            continue
        X_list.append(x)
        meta_list.append(pd.read_csv(csv_path))
    X = np.concatenate(X_list, axis=0)
    meta = pd.concat(meta_list, axis=0).reset_index(drop=True)
    return X, meta

if not TRAIN_SHARD_DIR.exists():
    raise FileNotFoundError(
        f"{TRAIN_SHARD_DIR} 가 없습니다.\n"
        "01_extract_training.py -> 02_common_preprocess.py -> "
        "03_extract_ecapa_embeddings.py를 먼저 실행해서 "
        "임베딩을 만들어야 이 노트북을 실행할 수 있습니다.\n"
        "(이미 실행해서 저장된 출력을 읽으시려면 이 셀을 실행하지 말고 "
        "저장된 출력을 그대로 확인하세요.)"
    )

X_all, meta_all = load_shards(TRAIN_SHARD_DIR)
train_mask = (meta_all["split"] == "train").to_numpy()
val_mask = (meta_all["split"] == "val").to_numpy()
X_train, meta_train = X_all[train_mask], meta_all[train_mask].reset_index(drop=True)
X_val, meta_val = X_all[val_mask], meta_all[val_mask].reset_index(drop=True)
y_train = (meta_train["gender"] == "F").astype(int).to_numpy()

X_pv, meta_pv = load_shards(PV_SHARD_DIR)

print("Train segment:", len(meta_train), " Val segment:", len(meta_val), " Data/Validation segment:", len(meta_pv))
print("Train 통화:", meta_train["call_id"].nunique(), " Val 통화:", meta_val["call_id"].nunique(),
      " Data/Validation 통화:", meta_pv["call_id"].nunique())


Train segment: 369037  Val segment: 93159  Data/Validation segment: 58102
Train 통화: 23360  Val 통화: 5840  Data/Validation 통화: 3640


## 2. 데이터 개요

- `Data/Training`(29,200통화)을 신고자 구간만 crop → 462,197개 segment (1개 추출 실패)
- 팀 고정 split: train 23,360통화 / val 5,840통화
- `Data/Validation`(대회 제공 별도 파티션, 3,640통화): 학습에는 전혀 사용하지 않고 평가 전용


In [2]:
print("성별 분포 (Train 통화 기준):")
print(meta_train.groupby("call_id")["gender"].first().value_counts())
print()
print("segment당 평균 duration(ms):", meta_train["duration_ms"].mean().round(1))
print("segment duration 범위:", meta_train["duration_ms"].min(), "~", meta_train["duration_ms"].max(), "ms")


성별 분포 (Train 통화 기준):
gender
F    12434
M    10926
Name: count, dtype: int64

segment당 평균 duration(ms): 2120.9
segment duration 범위: 70.0 ~ 24172.0 ms


## 3. 임베딩 추출: VRAM 병목과 해결

frozen ECAPA-TDNN으로 462,197개 segment의 임베딩을 추출하는 과정에서, 처리 속도가
초당 270개에서 초당 2개까지 떨어지는 문제가 있었습니다. `nvidia-smi`로 확인한 결과
VRAM 96% 사용, GPU 전력은 유휴 수준(15W/16W)이라는 모순된 지표로 원인을 특정했습니다.

원인은 segment를 duration 순으로 정렬해 패딩 낭비는 줄였지만, 배치 "개수"를 32개로
고정해둔 것이었습니다. 뒤로 갈수록 최대 24초짜리 오디오까지 32개씩 묶으면서 4GB VRAM이
거의 꽉 찼던 것입니다.

**해결**: 배치를 "개수"가 아니라 "배치 내 오디오 길이 합" 기준으로 동적으로 구성하도록
수정(`03_extract_ecapa_embeddings.py`). 남은 229,205개 처리 시간이 예상 42시간 →
실제 68분으로 단축(약 37배). 이 수정이 임베딩 값 자체에 영향을 주지 않는다는 것도
배치 구성을 바꿔가며 cosine similarity로 직접 검증했습니다(0.9999998 수준, 150ms 이상
정상 길이 segment 기준).


## 4. Baseline: Logistic Regression

192차원 임베딩을 표준화한 뒤 Logistic Regression으로 분류. 통화 단위 판단은 그 통화에
속한 segment들의 예측 확률을 평균 내서 결정합니다.


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

scaler_base = StandardScaler()
X_train_base = scaler_base.fit_transform(X_train)
X_val_base = scaler_base.transform(X_val)

clf_base = LogisticRegression(max_iter=2000, class_weight="balanced")
clf_base.fit(X_train_base, y_train)

val_pred_seg = clf_base.predict(X_val_base)
y_val = (meta_val["gender"] == "F").astype(int).to_numpy()
print("Segment 단위 Accuracy:", round(accuracy_score(y_val, val_pred_seg), 4))
print(classification_report(y_val, val_pred_seg, target_names=["M", "F"]))


Segment 단위 Accuracy: 0.8591
              precision    recall  f1-score   support

           M       0.86      0.83      0.84     43026
           F       0.86      0.89      0.87     50133

    accuracy                           0.86     93159
   macro avg       0.86      0.86      0.86     93159
weighted avg       0.86      0.86      0.86     93159



In [4]:
val_proba_base = clf_base.predict_proba(X_val_base)[:, 1]

def aggregate_mean(meta_df, proba, threshold=0.5):
    df = meta_df.copy()
    df["proba"] = proba
    call_proba = df.groupby("call_id")["proba"].mean()
    call_true = df.groupby("call_id")["gender"].first().map({"M": 0, "F": 1})
    call_proba = call_proba.reindex(call_true.index)
    return (call_proba >= threshold).astype(int), call_true

pred, true = aggregate_mean(meta_val, val_proba_base)
print("Baseline 통화 단위 Accuracy (val_calls):", round(accuracy_score(true, pred), 4))


Baseline 통화 단위 Accuracy (val_calls): 0.9772


## 5. 개선: L2 정규화 + MLP + duration 가중평균

이미 뽑아둔 임베딩 위에서 세 가지를 바꿔봤습니다.
- 임베딩 L2 정규화 (화자 임베딩에서 관례적으로 쓰는 전처리)
- 분류기를 Logistic Regression → MLP(은닉층 128개)로 교체 (가장 크게 기여)
- 통화 단위 집계를 단순평균 → duration 비례 가중평균으로 변경

70~120ms 같은 극단적으로 짧은 신고자 발화는 배치 구성과 무관하게 임베딩 자체가
불안정하다는 것을 확인했는데(정상 길이는 cosine similarity 0.9999998, 초단문은
0.55~0.86), duration 가중평균이 이런 노이즈성 segment의 영향력을 자동으로 낮춰줘서
별도 필터링 없이도 해결됐습니다.


In [5]:
from sklearn.preprocessing import normalize
from sklearn.neural_network import MLPClassifier

DURATION_WEIGHT_CAP_MS = 3000

def aggregate_duration_weighted(meta_df, proba, threshold=0.5):
    df = meta_df.copy()
    df["proba"] = proba
    w = df["duration_ms"].clip(upper=DURATION_WEIGHT_CAP_MS)
    df["w_proba"] = df["proba"] * w
    grouped = df.groupby("call_id").agg(
        w_sum=("w_proba", "sum"),
        w_total=("duration_ms", lambda s: s.clip(upper=DURATION_WEIGHT_CAP_MS).sum()),
    )
    call_proba = grouped["w_sum"] / grouped["w_total"]
    call_true = df.groupby("call_id")["gender"].first().map({"M": 0, "F": 1})
    call_proba = call_proba.reindex(call_true.index)
    return (call_proba >= threshold).astype(int), call_true, call_proba

X_train_l2 = normalize(X_train, norm="l2")
X_val_l2 = normalize(X_val, norm="l2")

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train_l2)
X_val_s = scaler.transform(X_val_l2)

clf_mlp = MLPClassifier(hidden_layer_sizes=(128,), activation="relu", solver="adam",
                         alpha=1e-4, max_iter=150, early_stopping=True,
                         n_iter_no_change=10, random_state=42)
clf_mlp.fit(X_train_s, y_train)
val_proba_mlp = clf_mlp.predict_proba(X_val_s)[:, 1]

pred_mlp, true_mlp, _ = aggregate_duration_weighted(meta_val, val_proba_mlp, threshold=0.5)
print("단일 MLP(threshold=0.5) 통화 단위 Accuracy:", round(accuracy_score(true_mlp, pred_mlp), 4))


단일 MLP(threshold=0.5) 통화 단위 Accuracy: 0.98


In [6]:
# 위 단일 MLP(seed=42)를 Data/Validation에도 그대로 적용 (재학습 없음, 이미 fit된 clf_mlp 재사용)
X_pv_l2 = normalize(X_pv, norm="l2")
X_pv_s = scaler.transform(X_pv_l2)
pv_proba_mlp = clf_mlp.predict_proba(X_pv_s)[:, 1]

pred_mlp_pv, true_mlp_pv, _ = aggregate_duration_weighted(meta_pv, pv_proba_mlp, threshold=0.5)
print("단일 MLP(threshold=0.5) 통화 단위 Accuracy (Data/Validation):", round(accuracy_score(true_mlp_pv, pred_mlp_pv), 4))

단일 MLP(threshold=0.5) 통화 단위 Accuracy (Data/Validation): 0.9799


### Probability Calibration (Platt scaling) 적용

val_calls(학습에 전혀 쓰이지 않은 held-out)의 통화 단위 확률/정답으로 Platt scaling(1-feature 로지스틱 회귀)을 학습해서 적용했습니다.

| | calibration 전 | calibration 후 |
|---|---|---|
| val_calls | 98.07% | 98.07% (학습에 쓰인 데이터라 참고용) |
| Data/Validation | 98.05% | **98.27%** |

Data/Validation은 calibration을 학습할 때 전혀 보지 않은 데이터인데, 여기서는 98.05%에서 98.27%로 올랐습니다(3,640통화 중 8통화 차이). 개선 폭이 작아 일반적인 성능 향상이라고 단정하기는 어렵지만, 성능이 나빠지지는 않았고 확률값을 보정하는 방식은 규정에서 허용하므로 최종 모델에 포함했습니다 (`04_final_train_ensemble.py`의 `USE_PROBABILITY_CALIBRATION = True`, ckpt에 `calibrator` 저장). Decision Threshold 자체는 calibration 전후 모두 0.5로 동일합니다.

## 6. 통계적 검증 

단일 숫자만으로는 근거가 약하다고 판단해서 세 가지를 추가로 확인했습니다.
- **Seed 안정성**: MLP를 6개 다른 seed로 반복 학습 → 표준편차 0.05~0.09%p로 매우 작음
- **McNemar 검정**: baseline과 짝지어 비교. **최종 채택한 10-seed 앙상블**은 두 검증셋
  모두 p<0.05로 유의미함 (`04_final_train_ensemble.py` 실행 로그, 7번 섹션 참고)
- **95% 신뢰구간**: Wilson score interval로 불확실성 범위 제시

아래 셀은 방법론을 보여주기 위한 축약본으로, **단일 seed(42) MLP 하나만** baseline과
비교합니다. seed 하나만 보면 아래처럼 통계적으로 유의미하지 않게 나올 수도 있는데,
이게 바로 단일 모델 하나를 고르지 않고 10개 seed를 평균 낸 앙상블을 최종 채택한
이유입니다 (7번 섹션).


In [7]:
from scipy.stats import binomtest

def mcnemar_exact(pred_a, pred_b, y_true):
    correct_a = (pred_a == y_true)
    correct_b = (pred_b == y_true)
    b = int(((~correct_a) & correct_b).sum())
    c = int((correct_a & (~correct_b)).sum())
    n = b + c
    if n == 0:
        return b, c, 1.0
    result = binomtest(min(b, c), n, 0.5, alternative="two-sided")
    return b, c, result.pvalue

# threshold는 0.5로 고정 
pred_mlp_thr, _, _ = aggregate_duration_weighted(meta_val, val_proba_mlp, 0.5)
b, c, p = mcnemar_exact(pred.to_numpy(), pred_mlp_thr.to_numpy(), true.to_numpy())
print(f"McNemar (baseline vs 개선): 고침 {b} / 새로틀림 {c}, p={p:.6f}")


McNemar (baseline vs 개선): 고침 45 / 새로틀림 29, p=0.080507


## 7. 최종 채택: 10-seed MLP 앙상블

같은 구성을 random seed 10개로 반복 학습해서 확률을 평균 냈습니다. 단일 최고 seed와
비교했을 때 정확도 차이는 통계적으로 유의미하지 않았지만(McNemar p=0.75), **단일
최고 seed는 여러 개 중 val_calls 성능이 제일 좋은 걸 나중에 고른 것이라 선택 편향
가능성이 있는 반면, 앙상블은 아무것도 선택하지 않고 전부 평균 낸 것이라 이런 편향에서
자유롭습니다.** 추론 비용 차이도 거의 없어서 최종적으로 앙상블을 채택했습니다.

(전체 10-seed 학습/비교 코드는 `Code/04_final_train_ensemble.py`에 있습니다. 여기서는
결과만 요약합니다.)

| 구성 | val_calls | Data/Validation |
|---|---|---|
| Baseline (raw+LogReg) | 97.72% | 97.64% |
| 단일 MLP(seed=42, 6번 섹션 축약 코드로 재현) | ~98.0%* | 97.99% |
| 10-seed 앙상블 (calibration 전) | 98.07% | 98.05% |
| **10-seed 앙상블 + calibration (최종, threshold=0.5)** | **98.07%** | **98.27%** |

*단일 seed 하나만의 결과라 재실행 시 소폭 달라질 수 있습니다. seed 안정성 확인용으로
6개 seed를 반복했을 때 표준편차는 0.05~0.09%p 수준이었습니다(6번 섹션 참고).

calibration 전후를 비교하면 val_calls는 98.07%로 변화가 없고, Data/Validation은 98.05%에서 98.27%로 올랐습니다(3,640통화 중 8통화 차이). calibration은 정확도가 아니라 확률값의 품질을 맞추는 단계라, 학습 분포와 같은 val_calls에서는 판정이 거의 바뀌지 않았습니다. Data/Validation에서는 앙상블 확률이 한쪽으로 약간 치우쳐 있던 것이 보정되면서 경계 근처 통화 일부가 바로잡힌 것으로 보입니다. 다만 개선 폭이 작아 일반적인 성능 향상이라고 단정하기는 어렵고, 최종 Decision Threshold는 규정에 따라 0.5로 고정했습니다.

## 8. 시도했지만 채택하지 않은 것들

정확도를 더 끌어올리기 위해 아래 방법들을 추가로 시도했지만, 전부 기존 최고 기록
(98.27%) 대비 통계적으로 유의미한 개선을 보이지 못했습니다. 

### 8-1. 이종 앙상블 (LogReg + MLP 스태킹)
서로 다른 결정경계를 가진 두 모델을 조합하면 보완될 것이라 기대했으나, 학습된
메타분류기가 LogReg 신호를 거의 무시하고 MLP만 사용하는 것으로 수렴 (계수: MLP +2.99,
LogReg −0.64). Data/Validation 98.19% (McNemar p=0.63, 유의미하지 않음).

### 8-2. F0(피치) 음향 특징 추가
192차원 임베딩에 F0 평균/중앙값/표준편차 3개를 추가해도 변화 없음 (Data/Validation
98.24%, McNemar p=1.0). ECAPA가 이미 피치 정보를 암묵적으로 충분히 담고 있다는 뜻으로
해석했습니다.


## 9. 오분류 원인 분석

Data/Validation에서 틀린 63개 통화(1.73%)를 직접 분석했습니다.

- 틀린 통화와 맞은 통화의 segment 개수/duration 분포는 거의 차이가 없었음 (정보 부족
  문제가 아님)
- 확신도 분포: 17개는 threshold 근처 애매한 케이스, **12개는 확신도 0.30 이상으로
  강하게 확신하고 틀림**. 이 패턴이 "라벨 자체의 모호성" 가설로 이어짐
- 발화 텍스트를 직접 읽어본 결과, 일부는 "제3자(가족, 시설 직원)가 환자를 대신해
  신고"하는 상황이었고, gender 라벨이 신고자 본인이 아니라 환자를 가리키는 것으로
  보이는 사례 발견
- 접수자가 제3자의 성별을 직접 묻는 패턴(Tier 1 신호)이 오분류군에서 28.6%, 정답군에서
  17.5% (Fisher's exact test, p=0.029, 통계적으로 유의미)
- 이 가설로 설명되는 오분류는 63개 중 약 7개(기대값 대비 초과분) 수준으로 추정.
  나머지는 설명되지 않는 어려운 케이스

이 분석은 오분류 call들의 segment 통계, 확신도 분포, 발화 텍스트, 접수자 질문 패턴을
단계적으로 뜯어보며 진행했습니다.

> **참고**: 이 분석은 calibration 적용 전 버전 모델의 오분류 63건을 대상으로 했습니다. 최종 모델(threshold=0.5 고정 + calibration)도 Data/Validation 정확도 98.27%, 오분류 63건(1.73%)으로 개수는 같지만, 정확히 같은 call인지는 별도로 대조하지 않았습니다. 위 정성적 결론(제3자 대리 신고로 인한 라벨 모호성)은 그대로 유효합니다.


## 10. 최종 결론

frozen ECAPA-TDNN 임베딩 기반 접근에서 추가로 시도한 저비용 개선(이종 앙상블 스태킹,
F0 음향 특징 추가)은 모두 통계적으로 유의미한 수준에 도달하지 못했습니다. 오분류
원인 분석 결과 일부(63개 중 약 7개)는 라벨 자체의 모호성(제3자 대리 신고)으로
설명되어, 이 접근법 범위 내에서는 **98.1~98.3% 수준이 실질적인 상한**이라고 판단했습니다.

최종 제출 모델: L2정규화 + MLP(hidden=128) 10-seed 앙상블 + duration 가중평균 집계 +
Platt scaling calibration(val_calls로 학습) + Decision Threshold 0.5 고정 (`Code/ecapa_ensemble_v3.ckpt`, `Code/inference.py`).

| 검증셋 | 통화 수 | 최종 정확도 |
|---|---|---|
| val_calls.csv (Training 파티션 내부 held-out) | 5,840 | 98.07% |
| Data/Validation (대회 제공 별도 파티션) | 3,640 | 98.27% |
